# ***PREPROCESAMIENTO Y FEATURE ENGINEERING***

### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from category_encoders.binary import BinaryEncoder

## *Preprocesamiento*

### Obtiene fecha y nombre del artículo con la columna url

In [ ]:
df_clean['url_cleaned'] = df_clean['url'].astype(str).str.strip()
commit_df_change(df_clean, "Create url_cleaned column")

# Usar regex para encontrar el patrón de fecha YYYY/MM/DD
date_pattern = r'/(\d{4})/(\d{2})/(\d{2})/'
date_match = df_clean['url_cleaned'].str.extract(date_pattern) # Extraer la fecha y guardarla en una variable temporal

df_clean['article_year'] = date_match[0]
df_clean['article_month'] = date_match[1]
df_clean['article_day'] = date_match[2]
commit_df_change(df_clean, "Create article year, month and day columns")

# Convertir columnas de fecha a numérico, forzando errores a NaN
df_clean['article_year'] = pd.to_numeric(df_clean['article_year'], errors='coerce')
df_clean['article_month'] = pd.to_numeric(df_clean['article_month'], errors='coerce')
df_clean['article_day'] = pd.to_numeric(df_clean['article_day'], errors='coerce')
commit_df_change(df_clean, "Force errors to NaN in date columns")

# Extraer título (última parte de la ruta URL antes de la barra final)
# Eliminar esquema, netloc y barra final, luego dividir por '/' y tomar la última parte
df_clean['article_title'] = df_clean['url_cleaned'].str.split('/').str[-2]
commit_df_change(df_clean, "Extract title from the last part of the URL")

# Modifica el nombre del artículo, inicia con mayúsculas y remueve guiones
df_clean['article_title'] = df_clean['article_title'].str.replace('-', ' ').str.title()
commit_df_change(df_clean, "Modified the article name")


print("Nuevas columnas 'article_year', 'article_month', 'article_day' y 'article_title' creadas y modificadas.")


display(df_clean[['url', 'article_year', 'article_month', 'article_day', 'article_title']].head())

### Trasnforma booleanas de días de la semana en una sola columna

In [ ]:
# Columnas booleanas para días de la semana
weekday_cols = ['weekday_is_monday', 'weekday_is_tuesday', 'weekday_is_wednesday',
                'weekday_is_thursday', 'weekday_is_friday', 'weekday_is_saturday',
                'weekday_is_sunday']
weekday_names = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']

# Crear la columna 'day_of_week'
def get_weekday_name(row):
    for i, col in enumerate(weekday_cols):
        if row[col] == 1:
            return weekday_names[i]
    return 'unknown' # Handle cases where no weekday is marked as 1

df_clean['day_of_week'] = df_clean[weekday_cols].apply(get_weekday_name, axis=1)
commit_df_change(df_clean, "Created Day of Week column")

print("Columna 'day_of_week' creada")

### Transforma booleanas de canales de datos en una sola columna

In [ ]:
# Columnas booleanas para canales de datos
channel_cols = ['data_channel_is_lifestyle', 'data_channel_is_entertainment',
                'data_channel_is_bus', 'data_channel_is_socmed',
                'data_channel_is_tech', 'data_channel_is_world']
channel_names = ['lifestyle', 'entertainment', 'bus', 'socmed', 'tech', 'world']

# Crear la columna 'data_channel'
def get_channel_name(row):
    for i, col in enumerate(channel_cols):
        if row[col] == 1:
            return channel_names[i]
    return 'no_channel' # Handle cases where no channel is marked as 1

df_clean['data_channel'] = df_clean[channel_cols].apply(get_channel_name, axis=1)
commit_df_change(df_clean, "Created data channel column")

print("Columna 'data_channel' creada")

### Mostrar las primeras filas del dataframe con las nuevas columnas

In [ ]:
print("\nPrimeras filas del dataframe con variables de día de semana y canal:")
display(df_clean[['day_of_week', 'data_channel']].head())

print("\nInformación del dataframe con las nuevas columnas:")
print(df_clean[['day_of_week', 'data_channel']].info())

print("\nFin del preprocesamiento")

## *Feature engeneering*

### Función para generar histograma y gráfico Q-Q para conocer comportamiento de las variables importantes

In [ ]:
def diagnostic_plots(df_clean, variable):
    plt.figure(figsize=(10,4))
    plt.subplot(1, 2, 1)
    df_clean[variable].hist(bins=30, color='gray', edgecolor='black')
    plt.title(f"Histogram of {variable}")
    plt.subplot(1, 2, 2)
    stats.probplot(df_clean[variable], dist="norm", plot=plt)
    plt.gca().get_lines()[0].set_color('gray')
    plt.gca().get_lines()[1].set_color('red')
    plt.title(f"Q-Q plot of {variable}")
    plt.show()

### Genera gráficos de comportamiento de las variables importantes

In [ ]:
for col in df_clean[important_numeric_cols]:
    diagnostic_plots(df_clean, col)

### Transformación de las variables numéricas con método Yeo Jhonson

In [ ]:
transformer = PowerTransformer(method="yeo-johnson", standardize=False)#Configuramos la transformación
transformer.fit(df_clean[important_numeric_cols])#Ajustamos el transformador
transf_df = pd.DataFrame(transformer.transform(df_clean[important_numeric_cols]), columns=important_numeric_cols, index=df_clean.index)#Transformamos cada variable del dataframe
print("Lambdas estimadas:", transformer.lambdas_)
print(transf_df.head())

### Histograma y gráfico Q-Q para observar efecto de la transformación

In [ ]:
for col in df_clean[important_numeric_cols]:
    diagnostic_plots(df_clean, col)

### Escalado de variables con min max


In [ ]:
scaler=MinMaxScaler()
minmax_df = pd.DataFrame(scaler.fit_transform(transf_df), columns=transf_df.columns, index=transf_df.index)

print(minmax_df.head())

### Graficación de variables escaladas

In [ ]:
fig, axes = plt.subplots(3,2, figsize=(12,10))
plt.subplots_adjust(wspace=.3, hspace=.6)

axes = axes.ravel()
for col, ax in zip(minmax_df.columns, axes):
  n, bins, edges = ax.hist(minmax_df[col], color='gray', bins=20, edgecolor='black', density=True)
  ax.set_xticks(bins)
  ax.tick_params(axis='x',rotation=90)
  ax.set(title=f'{col}', xlabel=None)

  x = np.linspace(np.min(minmax_df[col]), np.max(minmax_df[col]), 100)
  y = norm.pdf(x, np.mean(minmax_df[col]), np.std(minmax_df[col]))
  ax.plot(x, y, color='red')

### Renombra columnas de df min max agregando '_scaled' al final

In [ ]:
minmax_df.columns = [col + '_scaled' for col in minmax_df.columns]
minmax_df.head()

### Cardinalidad variables categoricas

In [ ]:
var_categoricas = df_clean.describe(include='object').columns

highly_cardinal_variables = [];
few_cardinal_variables = [];

for col in var_categoricas:
  df_clean[col].nunique()
  if df_clean[col].nunique() >= 100:
    highly_cardinal_variables.append(col)
  else:
    few_cardinal_variables.append(col)
print(f'Variables con alta cardinalidad: {highly_cardinal_variables}')
print(f'Variables con baja cardinalidad: {few_cardinal_variables}')

### Codificar variables categóricas con baja cardinalidad

In [ ]:
encoder = OneHotEncoder(drop='first')
encoded_data = encoder.fit_transform(df_clean[few_cardinal_variables])
onehot_df = pd.DataFrame(encoded_data.toarray(), columns=encoder.get_feature_names_out())
onehot_df

### Agrega variables escaladas al dataframe

In [ ]:
df_encoded = pd.concat([df_clean, minmax_df, onehot_df], axis=1)
commit_df_change(df_encoded, "Added scaled variables to df")

print('Fin del feature engineering')

df_encoded.head()